In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

model_name = "nlptown/bert-base-multilingual-uncased-sentiment"

print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# --- The pipeline way ---
# sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer, device=0 if torch.cuda.is_available() else -1)

# print("\n--- Sentiment Analysis Example (Pipeline Way) ---")

# text_positive_pipeline = "Tento film bol absolútne geniálny! Určite odporúčam." # Slovak positive (5 stars)
# prediction_positive_pipeline = sentiment_pipeline(text_positive_pipeline)
# print(f"\nInput: '{text_positive_pipeline}'")
# print(f"Prediction: {prediction_positive_pipeline}")

# text_negative_pipeline = "Tento servis bol hrozný, skutočné sklamanie." # Slovak negative (1 star)
# prediction_negative_pipeline = sentiment_pipeline(text_negative_pipeline)
# print(f"\nInput: '{text_negative_pipeline}'")
# print(f"Prediction: {prediction_negative_pipeline}")

# text_neutral_pipeline = "Jedlo bolo v poriadku, ale atmosféra ma veľmi neoslovila." # Slovak neutral/mixed (3 stars)
# prediction_neutral_pipeline = sentiment_pipeline(text_neutral_pipeline)
# print(f"\nInput: '{text_neutral_pipeline}'")
# print(f"Prediction: {prediction_neutral_pipeline}")


# --- The manual way ---
print("\n--- Manual Sentiment Analysis Example ---")

# Get label mapping from model config
id2label = model.config.id2label

# The 'nlptown/bert-base-multilingual-uncased-sentiment' model typically outputs star ratings (1 to 5)
# id2label_renamed will map the numerical class IDs (0-4) to their respective star ratings.
id2label_renamed = {0: '1 star', 1: '2 stars', 2: '3 stars', 3: '4 stars', 4: '5 stars'}

def get_manual_sentiment(text, model, tokenizer, device, label_map):
    # 1. Tokenize the input
    # Using the tokenizer directly as a callable is the recommended modern approach
    inputs = tokenizer(
        text,
        return_tensors="pt",
        add_special_tokens=True,
        truncation=True,
        padding='max_length',
        max_length=512
    ).to(device)

    # 2. Perform model inference
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits

    # 3. Apply softmax to get probabilities
    probabilities = torch.softmax(logits, dim=1)

    # 4. Get the predicted label and its score
    max_probability = torch.max(probabilities).item()
    predicted_class_id = torch.argmax(probabilities).item()
    predicted_label = label_map[predicted_class_id]

    return {'label': predicted_label, 'score': max_probability}

# Example 1: Very Positive sentiment (Slovak - manual) - aiming for 5 stars
text_positive_manual = "Tento film bol absolútne geniálny! Určite odporúčam."
prediction_positive_manual = get_manual_sentiment(text_positive_manual, model, tokenizer, device, id2label_renamed)
print(f"\nInput (manual way): '{text_positive_manual}'")
print(f"Prediction (manual way): {prediction_positive_manual}")

# Example 2: Very Negative sentiment (Slovak - manual) - aiming for 1 star
text_negative_manual = "Tento servis bol hrozný, skutočné sklamanie."
prediction_negative_manual = get_manual_sentiment(text_negative_manual, model, tokenizer, device, id2label_renamed)
print(f"\nInput (manual way): '{text_negative_manual}'")
print(f"Prediction (manual way): {prediction_negative_manual}")

# Example 3: Neutral/Mixed sentiment (Slovak - manual) - aiming for 3 stars
text_neutral_manual = "Jedlo bolo v poriadku, ale atmosféra ma veľmi neoslovila."
prediction_neutral_manual = get_manual_sentiment(text_neutral_manual, model, tokenizer, device, id2label_renamed)
print(f"\nInput (manual way): '{text_neutral_manual}'")
print(f"Prediction (manual way): {prediction_neutral_manual}")

Loading nlptown/bert-base-multilingual-uncased-sentiment...


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


--- Manual Sentiment Analysis Example ---

Input (manual way): 'Tento film bol absolútne geniálny! Určite odporúčam.'
Prediction (manual way): {'label': '5 stars', 'score': 0.7809332609176636}

Input (manual way): 'Tento servis bol hrozný, skutočné sklamanie.'
Prediction (manual way): {'label': '1 star', 'score': 0.8017141222953796}

Input (manual way): 'Jedlo bolo v poriadku, ale atmosféra ma veľmi neoslovila.'
Prediction (manual way): {'label': '2 stars', 'score': 0.5188552141189575}


In [ ]:
!rm -rf ~/.cache/huggingface
print("Hugging Face cache cleared.")

Hugging Face cache cleared.
